# 00 イントロ：Qwen3-4B とチャットしてみる

このノートブックでは **Qwen3-4B** を動かしながら、言語モデルの基本を体験します。

## このノートブックでやること

1. **環境確認** — デバイス (CPU / MPS / CUDA) とパッケージを確認する
2. **モデル読み込み** — Hugging Face から Tokenizer と Model を準備する
3. **チャット** — メッセージを送って返答を生成する
4. **マルチターン** — 複数のやりとりを続ける

## Qwen3-4B とは

- Alibaba が開発した **4B（40億）パラメータ**の言語モデル
- 中国語・英語・日本語など多言語対応
- **Thinking モード**（推論を段階的に考える）と **non-thinking モード** を切り替えられる
- このノートブックでは non-thinking モード（`enable_thinking=False`）を使う

## 全体の流れ（Transformer の処理）

```
テキスト
  ↓ tokenizer.apply_chat_template()  # チャット形式のフォーマットに変換
トークン列
  ↓ model.generate()                 # Transformer が次のトークンを繰り返し予測
生成トークン列
  ↓ tokenizer.decode()               # トークンをテキストに戻す
応答テキスト
```

---
## 1. 環境確認

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print("device:", device)
print("dtype:", dtype)

---
## 2. モデル読み込み

Tokenizer と Model を Hugging Face cache から読み込みます。  
初回は自動でダウンロードされます（約 8 GB）。  
2回目以降は cache から読むので高速です。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B"

print("tokenizer 読み込み中...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("  語彙サイズ:", tokenizer.vocab_size)
print("  tokenizer クラス:", type(tokenizer).__name__)

In [ ]:
print("model 読み込み中...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="eager",  # attention weights を取得できる実装
)
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"  パラメータ数: {total_params / 1e9:.2f}B")
print(f"  モデルクラス: {type(model).__name__}")
print(f"  レイヤー数: {model.config.num_hidden_layers}")
print(f"  隠れ層の次元数: {model.config.hidden_size}")
print(f"  attention head 数: {model.config.num_attention_heads}")

---
## 3. チャット関数を作る

`chat()` 関数を定義します。

処理の流れ：
1. `messages`（ユーザーとアシスタントのやりとりリスト）を **chat template** でフォーマット
2. テキストを **トークン列**（整数の配列）に変換
3. `model.generate()` で次のトークンを繰り返し予測 → 応答トークン列
4. 生成部分だけを取り出してテキストにデコード

In [ ]:
def chat(messages: list[dict], max_new_tokens: int = 256) -> str:
    """messages を受け取り、モデルの応答テキストを返す。"""
    # chat template を適用してプロンプト文字列を作る
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    # トークン化
    inputs = tokenizer(text, return_tensors="pt").to(device)
    n_input = inputs["input_ids"].shape[1]

    # 生成
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # 入力部分を除いた生成トークンだけをデコード
    generated_ids = output_ids[0][n_input:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response

---
## 4. チャット：シングルターン

1 往復のシンプルな会話から始めます。

In [ ]:
# シングルターンの例
messages = [
    {"role": "user", "content": "言語モデルとは何か、1〜2文で教えてください。"}
]

response = chat(messages)
print("[User]", messages[-1]["content"])
print("[Assistant]", response)

---
## 5. チャット：マルチターン

会話履歴（`messages` リスト）に応答を追加し続けることで、**文脈を保ったやりとり**ができます。

Transformer は入力トークン列全体を毎回処理するため、  
過去の発言をリストに入れて渡すだけで文脈が維持されます。

In [ ]:
def chat_turn(messages: list[dict], user_text: str, **kwargs) -> tuple[list[dict], str]:
    """user_text を追加して応答を得る。更新済み messages と応答を返す。"""
    messages = messages + [{"role": "user", "content": user_text}]
    response = chat(messages, **kwargs)
    messages = messages + [{"role": "assistant", "content": response}]
    return messages, response


def print_turn(user_text: str, response: str):
    print(f"[User]      {user_text}")
    print(f"[Assistant] {response}")
    print()

In [ ]:
# 最初のターン
history = []
history, resp = chat_turn(history, "トークンとは何ですか？")
print_turn(history[-2]["content"], resp)

In [ ]:
# 2ターン目：前の回答を受けて続ける
history, resp = chat_turn(history, "具体的に「東京」という単語はいくつのトークンに分割されますか？")
print_turn(history[-2]["content"], resp)

In [ ]:
# 3ターン目
history, resp = chat_turn(history, "ありがとう。では Qwen3 の語彙サイズはいくつですか？")
print_turn(history[-2]["content"], resp)

---
## 6. 参考：chat template の中身を見る

`apply_chat_template()` が何を作っているか確認します。  
Special token（`<|im_start|>` など）がどこに入るかを見てみましょう。

In [ ]:
sample_messages = [
    {"role": "user", "content": "こんにちは"},
    {"role": "assistant", "content": "こんにちは！何かお手伝いできますか？"},
    {"role": "user", "content": "今日の天気は？"},
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(repr(formatted))

In [ ]:
# トークン列も見てみる
token_ids = tokenizer.encode(formatted)
print(f"トークン数: {len(token_ids)}")
print()

for i, tid in enumerate(token_ids):
    piece = tokenizer.decode([tid])
    print(f"  [{i:2d}] id={tid:6d}  '{piece}'")

---
## まとめ

| ステップ | 処理 | 関数 |
|---|---|---|
| フォーマット | テキスト → chat template 形式 | `tokenizer.apply_chat_template()` |
| トークン化 | テキスト → token ID 列 | `tokenizer()` |
| 生成 | token ID 列 → 応答 token ID 列 | `model.generate()` |
| デコード | 応答 token ID 列 → テキスト | `tokenizer.decode()` |

次のノートブックでは **tokenizer** の動作を詳しく観察します。